In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# LLM 선언 - AzureOpenAI 로 활용해도 됨

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini") 
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.tools import tool


@tool
def add(a: int, b:int) -> int:
    """
        숫자 a 와 b를 더합니다    
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """
        숫자 a 와 b를 곱합니다    
    """
    return a * b


#### langgraph에서 ToolNode

In [4]:
from langgraph.prebuilt import ToolNode
print(ToolNode)

<class 'langgraph.prebuilt.tool_node.ToolNode'>


In [5]:
from langgraph.prebuilt import ToolNode


tool_list = [add, multiply] 

llm_with_tools = small_llm.bind_tools(tool_list)  # LLM 바인딩 
tool_node = ToolNode(tool_list)   # 랭그래프 바인딩 

In [6]:
query = '3 곱하기 5는?'

In [7]:
multiply.invoke({'a' : 3, 'b':5})  # <- 원래 사용하던 방식

15

In [8]:
# 이제는 tool_node에서 바로 invoke할 수 있다. 
ai_message = llm_with_tools.invoke("What is 3 plus 5?")
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 84, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DKiOSITVHhTtiPOSaqnHEwcccT17a', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d007b-670e-7121-956a-c0302ec0b4dd-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_nxZgWRadcHQNSldOJXvM1vEr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 17, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
# tool node는 list of messages, 즉, AnyMessage를 받는다. 
# 마지막 message는 AIMessage이어야 하고, tool call이 포함되어 있어야 한다. 

tool_node.invoke({'messages': [ai_message]}) # ToolNode는 messages를 받는다

# 결론
# messages라는 state를 가지고, List of AnyMessage(system, ai, human, tool) 의 형식을 띈다
# 그리고 마지막이 AIMessage이고, AImessage 가 Tool Calls를 갖는다. 

ValueError: Missing required config key 'N/A' for 'tools'.